# 08 — Panchang representation analysis

## Purpose

This notebook investigates how the five Panchang categories contribute to the short-term PV forecasts. It builds on the matched SOLETE development-fold predictions from Notebooks 06 and 07.

We compare the correctly aligned Panchang representation with circularly shifted and jointly permuted category sequences. We also examine reference-category coding, each Panchang component separately, and their combined forecasting effect.

We use the same LightGBM configuration, forecasting horizons and three chronological development folds throughout.

## 1. Experimental design

For each forecasting sample, **F3** includes 147 historical and continuous astronomical predictors. **F4** adds the 97 one-hot Panchang variables calculated from the same Sun–Moon geometry.

The model uses 300 LightGBM trees, learning rate 0.03 and 15 leaves. All comparisons use identical training and validation timestamps at horizons of 15, 30 and 60 minutes.

The control representations are:

| Representation | Construction |
|---|---|
| Aligned F4 | The five Panchang categories at the forecast-valid time |
| Shifted | Jointly circular-shift category rows by 17, 37 or 61 days within each training or validation fold |
| Permuted | Jointly permute category rows with seeds 1301, 1302 or 1303, separately within training and validation |
| Reference coding | Omit one fixed indicator from each category group |
| Single component | Add Tithi, Paksha, Karana, Nakshatra or Yoga alone to F3 |

Shifting preserves the ordering pattern away from the circular seam; permutation preserves categorical combinations and frequencies while changing their association with forecast-valid time.

In [1]:
from pathlib import Path
import gc
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    (
        directory for directory in (Path.cwd(), *Path.cwd().parents)
        if (directory / "solete_dataset" / "SOLETE_Pombo_5min.h5").is_file()
        and (directory / "results" / "feature_comparison" /
             "solete_development_predictions.csv.gz").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the SOLETE research project.")
PROJECT_ROOT = PROJECT_ROOT.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_solete
from src.preprocessing import build_cleaned_solete_features, build_gregorian_features
from src.astronomy_features import (
    build_lunar_f3_features,
    build_solar_f2_features,
    load_jpl_ephemeris,
    solete_astronomy_reference_times,
    tropical_longitudes_for_solete,
)
from src.vedic_calendar import build_panchang_f4_raw, lahiri_ayanamsha_for_utc_times
from src.feature_families import encode_panchang_fixed_vocabulary, panchang_group_columns
from src.forecasting_protocol import (
    CV_FOLDS,
    build_f0_history,
    build_forecast_targets,
    build_base_modelling_tables,
    make_rolling_origin_masks,
    validate_rolling_origin_counts,
)
from src.classical_models import build_lightgbm_model

HORIZONS = (15, 30, 60)
FOLDS = (1, 2, 3)
MODEL_CONFIG = {"n_estimators": 300, "learning_rate": 0.03, "num_leaves": 15}
SHIFT_DAYS = (17, 37, 61)
PERMUTATION_SEEDS = (1301, 1302, 1303)
PANCHANG_COMPONENTS = ("tithi", "paksha", "karana", "nakshatra", "yoga")
VARIANTS = (
    *(f"shift_{days}d" for days in SHIFT_DAYS),
    *(f"permutation_{seed}" for seed in PERMUTATION_SEEDS),
    "reference_coding",
    *(f"only_{name}" for name in PANCHANG_COMPONENTS),
)
METHODS = ("F3", "F4", *VARIANTS)
assert len(VARIANTS) == 12 and len(METHODS) == 14

SOURCE_RESULTS = PROJECT_ROOT / "results" / "feature_comparison"
recorded = pd.read_csv(
    SOURCE_RESULTS / "solete_development_predictions.csv.gz",
    parse_dates=["issue_time_UTC", "valid_time_UTC"],
)
assert len(recorded) == 157_212
assert set(recorded["horizon_min"]) == set(HORIZONS)
assert {"F3", "F4", "actual_kw", "daylight"}.issubset(recorded.columns)
assert recorded.duplicated(["horizon_min", "issue_time_UTC"]).sum() == 0
assert np.isfinite(recorded[["F3", "F4", "actual_kw"]].to_numpy()).all()
print("Matched development predictions:", len(recorded))
print("Additional Panchang representations:", len(VARIANTS))

Matched development predictions: 157212
Additional Panchang representations: 12


## 2. Reconstruct the astronomical and Panchang feature blocks

The preprocessing and astronomical calculations follow the definitions validated in Notebooks 02–05. The 5-minute SOLETE row uses its recorded UTC timestamp for indexing and the timestamp plus two minutes for solar and lunar position calculations.

The fixed Panchang vocabulary contains 30 Tithis, 2 Pakshas, 11 Karanas, 27 Nakshatras and 27 Yogas.

In [2]:
DATA_FILE = PROJECT_ROOT / "solete_dataset" / "SOLETE_Pombo_5min.h5"
EPHEMERIS_FILE = PROJECT_ROOT / "ephemeris" / "de440s.bsp"
if not EPHEMERIS_FILE.is_file():
    raise FileNotFoundError(f"Expected ephemeris: {EPHEMERIS_FILE}")

raw = load_solete(DATA_FILE).sort_index()
cleaned = build_cleaned_solete_features(raw)
times = cleaned.index
history = build_f0_history(cleaned)
base_tables = build_base_modelling_tables(
    cleaned,
    f0_history=history,
    target_tables=build_forecast_targets(cleaned),
)

calendar = build_gregorian_features(times)
solar = build_solar_f2_features(times)
ephemeris = load_jpl_ephemeris(EPHEMERIS_FILE)
lunar_pieces, longitude_pieces = [], []
BATCH_SIZE = 8192
for start in range(0, len(times), BATCH_SIZE):
    part = times[start:start + BATCH_SIZE]
    lunar_pieces.append(build_lunar_f3_features(part, ephemeris))
    longitude_pieces.append(tropical_longitudes_for_solete(part, ephemeris))
lunar = pd.concat(lunar_pieces)
tropical = pd.concat(longitude_pieces)
del lunar_pieces, longitude_pieces, ephemeris
gc.collect()

ayanamsha = pd.Series(
    lahiri_ayanamsha_for_utc_times(solete_astronomy_reference_times(times)),
    index=times,
)
raw_panchang = build_panchang_f4_raw(
    tropical["sun_tropical_longitude_deg"],
    tropical["moon_tropical_longitude_deg"],
    ayanamsha,
)
raw_panchang.columns = [f"valid_{column}" for column in raw_panchang.columns]
panchang_indicators = encode_panchang_fixed_vocabulary(raw_panchang)
del tropical, ayanamsha, raw_panchang
gc.collect()

assert [x.shape[1] for x in (history, calendar, solar, lunar, panchang_indicators)] == [118, 15, 5, 9, 97]
assert all(block.index.equals(times) for block in (history, calendar, solar, lunar, panchang_indicators))
assert (panchang_indicators.sum(axis=1) == 5).all()

PANCHANG_GROUPS = panchang_group_columns(panchang_indicators.columns)
GROUP_POSITIONS = {
    name: np.array([panchang_indicators.columns.get_loc(col) for col in cols])
    for name, cols in PANCHANG_GROUPS.items()
}
assert {name: len(idx) for name, idx in GROUP_POSITIONS.items()} == {
    "tithi": 30, "paksha": 2, "karana": 11, "nakshatra": 27, "yoga": 27,
}
REFERENCE_KEEP = np.array([
    position
    for name in PANCHANG_COMPONENTS
    for position in GROUP_POSITIONS[name][1:]
])
assert len(REFERENCE_KEEP) == 92

print("Feature dimensions:")
display(pd.DataFrame({
    "group": ["F3", "Aligned F4", "Reference-coded F4", *PANCHANG_COMPONENTS],
    "predictors": [147, 244, 239, *(147 + len(GROUP_POSITIONS[n]) for n in PANCHANG_COMPONENTS)],
}))

Feature dimensions:


,group,predictors
0,F3,147
1,Aligned F4,244
2,Reference-coded F4,239
3,tithi,177
4,paksha,149
5,karana,158
6,nakshatra,174
7,yoga,174


## 3. Build the aligned F3 and Panchang matrices

Each model uses the same issue-time observational history. Gregorian, solar, lunar and Panchang values are selected at the forecast-valid time. We assemble one horizon at a time to keep the intermediate feature matrices manageable.

In [3]:
def align_at_valid_time(block, base):
    valid_index = pd.DatetimeIndex(base["valid_time"])
    aligned = block.loc[valid_index]
    assert len(aligned) == len(base)
    return aligned.to_numpy(dtype=np.float32)


def horizon_feature_matrices(horizon):
    base = base_tables[horizon]
    f3 = np.column_stack([
        history.loc[base.index].to_numpy(dtype=np.float32),
        align_at_valid_time(calendar, base),
        align_at_valid_time(solar, base),
        align_at_valid_time(lunar, base),
    ])
    panchang = align_at_valid_time(panchang_indicators, base).astype(np.int8)
    assert f3.shape == (len(base), 147)
    assert panchang.shape == (len(base), 97)
    assert np.isfinite(f3).all()
    assert np.all(panchang.sum(axis=1) == 5)
    return f3, panchang


def variant_blocks(name, train_block, validation_block):
    """Apply one fixed coding or split-wise timing control."""
    if name.startswith("shift_"):
        days = int(name.removeprefix("shift_").removesuffix("d"))
        steps = days * 288
        return (
            np.roll(train_block, shift=steps, axis=0),
            np.roll(validation_block, shift=steps, axis=0),
        )
    if name.startswith("permutation_"):
        seed = int(name.removeprefix("permutation_"))
        train_rng = np.random.default_rng(seed)
        validation_rng = np.random.default_rng(seed + 1)
        return (
            train_block[train_rng.permutation(len(train_block))],
            validation_block[validation_rng.permutation(len(validation_block))],
        )
    if name == "reference_coding":
        return train_block[:, REFERENCE_KEEP], validation_block[:, REFERENCE_KEEP]
    if name.startswith("only_"):
        positions = GROUP_POSITIONS[name.removeprefix("only_")]
        return train_block[:, positions], validation_block[:, positions]
    raise ValueError(f"Unknown representation: {name}")

# Verify categorical row structure, grouping and reproducibility.
example = panchang_indicators.iloc[:200].to_numpy(dtype=np.int8)
shifted, _ = variant_blocks("shift_17d", example, example)
assert np.array_equal(shifted.sum(axis=1), example.sum(axis=1))
permuted, permuted_again = variant_blocks("permutation_1301", example, example)
np.testing.assert_array_equal(permuted, variant_blocks("permutation_1301", example, example)[0])
assert np.array_equal(permuted.sum(axis=0), example.sum(axis=0))
assert np.array_equal(permuted_again.sum(axis=0), example.sum(axis=0))
reference, _ = variant_blocks("reference_coding", example, example)
assert reference.shape[1] == 92
print("Representation control checks passed.")

Representation control checks passed.


## 4. Fit the timing, coding and single-component controls

We fit each control with the fixed LightGBM configuration on the three chronological development folds. The saved F3 and aligned-F4 predictions from Notebook 06 serve as matched references.

For each fold, the shifted and permuted sequences are constructed separately for training and validation. The selected blocks are inserted into the model alongside the unchanged 147 F3 predictors.

The output tables retain fold-level MAE and RMSE, plus UTC-day error totals for paired uncertainty analysis.

In [4]:
def collect_errors(metric_rows, daily_rows, *, horizon, fold, method,
                   valid_time, actual, prediction, daylight):
    actual = np.asarray(actual, dtype=np.float64)
    prediction = np.asarray(prediction, dtype=np.float64)
    daylight = np.asarray(daylight, dtype=bool)
    assert len(actual) == len(prediction) == len(daylight) == len(valid_time)
    assert np.isfinite(prediction).all()
    absolute_error = np.abs(actual - prediction)
    squared_error = np.square(actual - prediction)
    utc_days = pd.DatetimeIndex(valid_time).floor("D")
    for scope, mask in (
        ("all", np.ones(len(actual), dtype=bool)),
        ("daylight", daylight),
    ):
        assert mask.any()
        metric_rows.append({
            "horizon_min": horizon, "fold": fold, "scope": scope,
            "method": method, "n": int(mask.sum()),
            "mae_kw": float(absolute_error[mask].mean()),
            "rmse_kw": float(np.sqrt(squared_error[mask].mean())),
        })
        daily = pd.DataFrame({
            "utc_day": utc_days[mask],
            "absolute_error_sum_kw": absolute_error[mask],
            "squared_error_sum_kw2": squared_error[mask],
            "n": np.ones(int(mask.sum()), dtype=np.int32),
        }).groupby("utc_day", as_index=False).sum()
        daily["horizon_min"] = horizon
        daily["fold"] = fold
        daily["scope"] = scope
        daily["method"] = method
        daily_rows.append(daily)


metric_rows, daily_rows = [], []
models_fitted = 0

for horizon in HORIZONS:
    base = base_tables[horizon]
    assert validate_rolling_origin_counts(base["valid_time"], horizon)
    f3_full, panchang_full = horizon_feature_matrices(horizon)
    actual_full = base["target_P_Solar_kW"].to_numpy(dtype=np.float64)
    folds = make_rolling_origin_masks(base["valid_time"])

    for specification, masks in zip(CV_FOLDS, folds, strict=True):
        fold = specification["fold"]
        train = masks["train_mask"].to_numpy(dtype=bool)
        validation = masks["validation_mask"].to_numpy(dtype=bool)
        assert base["valid_time"].loc[train].max() < base["issue_time"].loc[validation].min()

        saved = recorded.loc[
            recorded["horizon_min"].eq(horizon) & recorded["fold"].eq(fold)
        ].sort_values("issue_time_UTC").reset_index(drop=True)
        expected_issue = pd.DatetimeIndex(base.index[validation])
        assert pd.DatetimeIndex(saved["issue_time_UTC"]).equals(expected_issue)
        assert pd.DatetimeIndex(saved["valid_time_UTC"]).equals(
            pd.DatetimeIndex(base["valid_time"].loc[validation])
        )
        actual_validation = actual_full[validation]
        np.testing.assert_allclose(
            saved["actual_kw"], actual_validation, rtol=0, atol=1e-8,
        )
        expected_daylight = align_at_valid_time(
            solar[["solar_daylight"]], base
        )[validation, 0].astype(np.int8)
        np.testing.assert_array_equal(saved["daylight"], expected_daylight)
        valid_time = saved["valid_time_UTC"]

        # References from the matched LightGBM development experiment.
        for method in ("F3", "F4"):
            collect_errors(
                metric_rows, daily_rows, horizon=horizon, fold=fold,
                method=method, valid_time=valid_time,
                actual=actual_validation, prediction=saved[method],
                daylight=expected_daylight,
            )

        train_parent = f3_full[train]
        valid_parent = f3_full[validation]
        train_panchang = panchang_full[train]
        valid_panchang = panchang_full[validation]
        assert (train_panchang.sum(axis=1) == 5).all()
        assert (valid_panchang.sum(axis=1) == 5).all()

        # Verify one original fitted model against its saved predictions.
        if horizon == 15 and fold == 1:
            canonical = build_lightgbm_model(MODEL_CONFIG)
            canonical.fit(
                np.concatenate([train_parent, train_panchang], axis=1),
                actual_full[train],
            )
            reestimated = canonical.predict(
                np.concatenate([valid_parent, valid_panchang], axis=1)
            )
            np.testing.assert_allclose(reestimated, saved["F4"], rtol=0, atol=1e-7)
            del canonical, reestimated
            print("Aligned-F4 parity check passed.")

        for variant in VARIANTS:
            p_train, p_valid = variant_blocks(
                variant, train_panchang, valid_panchang,
            )
            x_train = np.concatenate([train_parent, p_train], axis=1)
            x_valid = np.concatenate([valid_parent, p_valid], axis=1)
            assert x_train.shape[1] == x_valid.shape[1]
            assert np.isfinite(x_train).all() and np.isfinite(x_valid).all()

            model = build_lightgbm_model(MODEL_CONFIG)
            model.fit(x_train, actual_full[train])
            estimate = model.predict(x_valid)
            collect_errors(
                metric_rows, daily_rows, horizon=horizon, fold=fold,
                method=variant, valid_time=valid_time,
                actual=actual_validation, prediction=estimate,
                daylight=expected_daylight,
            )
            models_fitted += 1
            del model, estimate, p_train, p_valid, x_train, x_valid
        print(f"{horizon}-minute horizon, fold {fold}: {len(VARIANTS)} controls scored")
        gc.collect()

    del f3_full, panchang_full
    gc.collect()

metrics = pd.DataFrame(metric_rows)
daily_errors = pd.concat(daily_rows, ignore_index=True)
assert models_fitted == 3 * 3 * len(VARIANTS) == 108
assert len(metrics) == 3 * 3 * 2 * len(METHODS) == 252
assert metrics.duplicated(["horizon_min", "fold", "scope", "method"]).sum() == 0
assert daily_errors["n"].gt(0).all()
assert daily_errors["absolute_error_sum_kw"].ge(0).all()
assert daily_errors.duplicated(["horizon_min", "fold", "scope", "method", "utc_day"]).sum() == 0
print("Models fitted:", models_fitted)
print("Fold-level metric records:", len(metrics))
print("Daily error records:", len(daily_errors))

Aligned-F4 parity check passed.
15-minute horizon, fold 1: 12 controls scored
15-minute horizon, fold 2: 12 controls scored
15-minute horizon, fold 3: 12 controls scored
30-minute horizon, fold 1: 12 controls scored
30-minute horizon, fold 2: 12 controls scored
30-minute horizon, fold 3: 12 controls scored
60-minute horizon, fold 1: 12 controls scored
60-minute horizon, fold 2: 12 controls scored
60-minute horizon, fold 3: 12 controls scored
Models fitted: 108
Fold-level metric records: 252
Daily error records: 15288


## 5. Compare the representations

For each horizon, we average MAE across the three validation folds with equal fold weights. We report **F3 MAE minus augmented MAE** to measure the change from adding the indicated representation. We also compare each disrupted or alternative encoding directly with aligned F4.

In [5]:
mean_scores = (
    metrics.groupby(["horizon_min", "scope", "method"], as_index=False)
    .agg(mean_mae_kw=("mae_kw", "mean"), mean_rmse_kw=("rmse_kw", "mean"))
)
score_table = mean_scores.pivot_table(
    index=["horizon_min", "scope"], columns="method", values="mean_mae_kw",
)
summary_rows = []
for (horizon, scope), values in score_table.iterrows():
    for method in METHODS:
        summary_rows.append({
            "horizon_min": horizon,
            "scope": scope,
            "method": method,
            "mean_fold_mae_kw": float(values[method]),
            "mae_reduction_vs_F3_kw": float(values["F3"] - values[method]),
            "aligned_F4_advantage_kw": float(values[method] - values["F4"]),
        })
summary = pd.DataFrame(summary_rows)
print("All-hours mean-fold MAE — aligned and timing controls:")
selected_methods = ["F3", "F4", "shift_17d", "shift_37d", "shift_61d",
                    "permutation_1301", "permutation_1302", "permutation_1303",
                    "reference_coding"]
display(score_table.xs("all", level="scope")[selected_methods].round(5))
print("Single Panchang component additions to F3:")
display(summary.loc[
    summary["scope"].eq("all") & summary["method"].str.startswith("only_")
].pivot(index="horizon_min", columns="method", values="mae_reduction_vs_F3_kw").round(5))

All-hours mean-fold MAE — aligned and timing controls:


method,F3,F4,shift_17d,shift_37d,shift_61d,permutation_1301,permutation_1302,permutation_1303,reference_coding
horizon_min,,,,,,,,,
15,0.18081,0.17919,0.18091,0.18138,0.18048,0.18041,0.18087,0.18080,0.17993
30,0.22278,0.22190,0.22219,0.22252,0.22042,0.22134,0.22115,0.22139,0.22259
60,0.28686,0.28545,0.28820,0.28504,0.28556,0.28500,0.28504,0.28513,0.28659


Single Panchang component additions to F3:


method,only_karana,only_nakshatra,only_paksha,only_tithi,only_yoga
horizon_min,,,,,
15,0.00120,0.00113,0.00046,0.00076,0.00088
30,0.00081,0.00115,-0.00212,0.00194,0.00289
60,0.00252,-0.00084,0.00414,0.00093,0.00028


## 6. Estimate paired uncertainty for the control comparisons

We resample complete UTC days within each chronological validation fold and combine the three fold results with equal weights. Each draw uses the same sampled days for aligned F4 and the comparison representation.

The positive sign of **MAE(control) − MAE(aligned F4)** indicates that aligned F4 has lower forecast error. The 95% intervals describe the variation induced by daily resampling of the fixed development-fold predictions.

In [6]:
BOOTSTRAP_REPETITIONS = 10_000
rng = np.random.default_rng(2026)
interval_rows = []

for horizon in HORIZONS:
    for scope in ("all", "daylight"):
        sampled_scores_by_fold = []
        for fold in FOLDS:
            records = daily_errors.loc[
                daily_errors["horizon_min"].eq(horizon)
                & daily_errors["scope"].eq(scope)
                & daily_errors["fold"].eq(fold)
            ]
            sums = records.pivot(
                index="utc_day", columns="method", values="absolute_error_sum_kw"
            ).sort_index()
            counts_by_method = records.pivot(
                index="utc_day", columns="method", values="n"
            ).sort_index()
            assert sums.notna().all().all()
            assert counts_by_method.notna().all().all()
            assert (counts_by_method.eq(counts_by_method["F4"], axis=0)).all().all()
            assert set(sums.columns) == set(METHODS)

            counts = counts_by_method["F4"].to_numpy(dtype=float)
            n_days = len(sums)
            assert n_days >= 20
            drawn = rng.integers(0, n_days, size=(BOOTSTRAP_REPETITIONS, n_days))
            denominators = counts[drawn].sum(axis=1)
            sampled_scores = {
                method: sums[method].to_numpy(dtype=float)[drawn].sum(axis=1)
                / denominators
                for method in METHODS
            }
            sampled_scores_by_fold.append(sampled_scores)

        for method in VARIANTS:
            deltas = [
                sample[method] - sample["F4"] for sample in sampled_scores_by_fold
            ]
            bootstrap_delta = np.mean(np.stack(deltas, axis=0), axis=0)
            lower, upper = np.quantile(bootstrap_delta, [0.025, 0.975])
            observed = float(score_table.loc[(horizon, scope), method]
                             - score_table.loc[(horizon, scope), "F4"])
            interval_rows.append({
                "horizon_min": horizon,
                "scope": scope,
                "control": method,
                "aligned_F4_advantage_kw": observed,
                "ci_lower_kw": float(lower),
                "ci_upper_kw": float(upper),
                "classification": (
                    "Aligned F4 lower MAE" if lower > 0 else
                    "Control lower MAE" if upper < 0 else "Unresolved"
                ),
            })

intervals = pd.DataFrame(interval_rows)
assert len(intervals) == 3 * 2 * len(VARIANTS) == 72
assert (intervals["ci_lower_kw"] <= intervals["ci_upper_kw"]).all()
assert np.isfinite(intervals[["aligned_F4_advantage_kw", "ci_lower_kw", "ci_upper_kw"]].to_numpy()).all()
print("Aligned F4 versus timing controls — all hours:")
display(intervals.loc[
    intervals["scope"].eq("all")
    & (intervals["control"].str.startswith("shift_")
       | intervals["control"].str.startswith("permutation_"))
].round(5))

Aligned F4 versus timing controls — all hours:


,horizon_min,scope,control,aligned_F4_advantage_kw,ci_lower_kw,ci_upper_kw,classification
0,15,all,shift_17d,0.00173,0.00091,0.00259,Aligned F4 lower MAE
1,15,all,shift_37d,0.00219,0.00135,0.00308,Aligned F4 lower MAE
2,15,all,shift_61d,0.00129,0.00047,0.00214,Aligned F4 lower MAE
3,15,all,permutation_1301,0.00122,0.00051,0.00197,Aligned F4 lower MAE
4,15,all,permutation_1302,0.00169,0.00087,0.00253,Aligned F4 lower MAE
5,15,all,permutation_1303,0.00162,0.00085,0.00239,Aligned F4 lower MAE
24,30,all,shift_17d,0.00029,-0.00125,0.00181,Unresolved
25,30,all,shift_37d,0.00062,-0.00087,0.00215,Unresolved
26,30,all,shift_61d,-0.00148,-0.00276,-0.00023,Control lower MAE
27,30,all,permutation_1301,-0.00055,-0.00171,0.00059,Unresolved


## 7. Examine component non-additivity

We compare the combined F4 MAE reduction with the sum of the reductions from the five single-component models. Their difference describes non-additivity in predictive performance under LightGBM fitting and MAE evaluation.

In [7]:
interaction_rows = []
for (horizon, scope), row in score_table.iterrows():
    combined_gain = float(row["F3"] - row["F4"])
    component_gain = {
        family: float(row["F3"] - row[f"only_{family}"])
        for family in PANCHANG_COMPONENTS
    }
    sum_component_gains = sum(component_gain.values())
    interaction_rows.append({
        "horizon_min": horizon,
        "scope": scope,
        "combined_F4_gain_kw": combined_gain,
        "sum_single_component_gains_kw": sum_component_gains,
        "nonadditivity_gap_kw": combined_gain - sum_component_gains,
        **{f"{name}_gain_kw": value for name, value in component_gain.items()},
    })
nonadditivity = pd.DataFrame(interaction_rows)
assert len(nonadditivity) == 6
print("Panchang combined and component additions:")
display(nonadditivity.round(5))

Panchang combined and component additions:


,horizon_min,scope,combined_F4_gain_kw,sum_single_component_gains_kw,nonadditivity_gap_kw,tithi_gain_kw,paksha_gain_kw,karana_gain_kw,nakshatra_gain_kw,yoga_gain_kw
0,15,all,0.00162,0.00443,-0.00280,0.00076,0.00046,0.00120,0.00113,0.00088
1,15,daylight,0.00322,0.00803,-0.00481,0.00144,0.00034,0.00271,0.00195,0.00159
2,30,all,0.00088,0.00467,-0.00379,0.00194,-0.00212,0.00081,0.00115,0.00289
3,30,daylight,0.00173,0.00792,-0.00619,0.00376,-0.00360,0.00100,0.00232,0.00443
4,60,all,0.00142,0.00705,-0.00563,0.00093,0.00414,0.00252,-0.00084,0.00028
5,60,daylight,0.00381,0.00507,-0.00125,-0.00118,0.00827,0.00317,-0.00406,-0.00113


## 8. Save the representation-control results

The output files contain the fold-level forecasting scores, UTC-day error totals, paired uncertainty intervals and component non-additivity summaries.

In [8]:
OUTPUT_DIR = PROJECT_ROOT / "results" / "representation_controls"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
paths = {
    "metrics": OUTPUT_DIR / "solete_panchang_control_fold_metrics.csv",
    "daily": OUTPUT_DIR / "solete_panchang_control_daily_errors.csv.gz",
    "summary": OUTPUT_DIR / "solete_panchang_control_summary.csv",
    "intervals": OUTPUT_DIR / "solete_panchang_control_intervals.csv",
    "nonadditivity": OUTPUT_DIR / "solete_panchang_nonadditivity.csv",
}
metrics.to_csv(paths["metrics"], index=False, float_format="%.12g")
daily_errors.to_csv(paths["daily"], index=False, compression="gzip", float_format="%.12g")
summary.to_csv(paths["summary"], index=False, float_format="%.12g")
intervals.to_csv(paths["intervals"], index=False, float_format="%.12g")
nonadditivity.to_csv(paths["nonadditivity"], index=False, float_format="%.12g")

assert len(pd.read_csv(paths["metrics"])) == 252
assert len(pd.read_csv(paths["daily"])) == len(daily_errors)
assert len(pd.read_csv(paths["summary"])) == 84
assert len(pd.read_csv(paths["intervals"])) == 72
assert len(pd.read_csv(paths["nonadditivity"])) == 6
for path in paths.values():
    print(path.relative_to(PROJECT_ROOT))
print("NOTEBOOK 08: PANCHANG REPRESENTATION CHECKS PASSED")

results/representation_controls/solete_panchang_control_fold_metrics.csv
results/representation_controls/solete_panchang_control_daily_errors.csv.gz
results/representation_controls/solete_panchang_control_summary.csv
results/representation_controls/solete_panchang_control_intervals.csv
results/representation_controls/solete_panchang_nonadditivity.csv
NOTEBOOK 08: PANCHANG REPRESENTATION CHECKS PASSED


## Interpretation

The timing controls compare the aligned Panchang sequence with representations that retain categorical structure but change its association with the forecasting timestamps. The reference-coded and single-component models show how predictive performance changes under alternative encodings and individual Panchang additions.

The reported uncertainty intervals use paired UTC-day resampling of frozen fitted forecasts. We interpret the collection of control comparisons as exploratory evidence about representation and model sensitivity. The combined-versus-component comparison quantifies non-additivity of model performance rather than isolating a physical interaction.